# Train Binary Toxicity Model cho ViCTSD

Dữ liệu ViCTSD có cột `Comment` và `Toxicity`. Notebook này train mô hình nhị phân:

```text
0 = clean
1 = toxic
```

Không dùng 3 lớp `clean/offensive/hate` khi chỉ có ViCTSD, vì ViCTSD không có nhãn hate riêng.

Output: `models/toxicity_binary_phobert/final_model/`.

In [1]:
!pip install pandas scikit-learn  transformers datasets accelerate tqdm -q

In [2]:
import re, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

TOXICITY_DATA_DIR = Path("data/raw/main_train/04_negative_attack_toxicity")
LABELED_DATA_DIRS = [Path("data/processed/toxicity"), Path("data/raw/custom_labeled/toxicity")]
OUTPUT_DIR = Path("models/toxicity_binary_phobert")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "vinai/phobert-base"
MAX_LENGTH = 256
TEST_SIZE = 0.15
VALID_SIZE = 0.15
RANDOM_SEED = 42
NUM_EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
MAX_ROWS_PER_FILE = None
USE_SYNTHETIC_CSKH = True
id2label = {0: "clean", 1: "toxic"}
label2id = {"clean": 0, "toxic": 1}
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_SEED)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
print("TOXICITY_DATA_DIR:", TOXICITY_DATA_DIR)
print("MODEL_NAME:", MODEL_NAME)

d:\learning\CareScore-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
TOXICITY_DATA_DIR: data\raw\main_train\04_negative_attack_toxicity
MODEL_NAME: vinai/phobert-base


In [3]:
def normalize_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " <url> ", text)
    text = re.sub(r"\S+@\S+", " <email> ", text)
    text = re.sub(r"\b\d{9,11}\b", " <phone> ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def read_csv_robust(csv_path):
    try: return pd.read_csv(csv_path)
    except Exception: return pd.read_csv(csv_path, engine="python", on_bad_lines="skip", encoding="utf-8")

def map_toxicity_label(value):
    if pd.isna(value): return None
    raw = str(value).strip().lower()
    clean = ["clean","normal","non-toxic","nontoxic","not_toxic","not toxic","safe","neutral","none","0","không độc hại","khong doc hai","không công kích","khong cong kich"]
    toxic = ["toxic","offensive","hate","hate speech","hatespeech","negative","abusive","insult","insulting","profane","attack","harassment","threat","1","2","3","công kích","cong kich","xúc phạm","xuc pham","tiêu cực","tieu cuc","độc hại","doc hai","thù ghét","thu ghet"]
    if raw in clean: return 0
    if raw in toxic: return 1
    try:
        n = int(float(raw))
        return 0 if n == 0 else 1
    except Exception:
        return None

TEXT_COL_CANDIDATES = ["Comment","comment","text","sentence","content","utterance","review","message","input","output","conversation","conversation_text"]
LABEL_COL_CANDIDATES = ["Toxicity","toxicity","toxicity_label","toxic","label","labels","class","target","category"]
def find_column(columns, candidates):
    lower = {str(c).lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower: return lower[cand.lower()]
    return None

def build_toxicity_input(row):
    for col in ["conversation_text","conversation","dialogue","dialog","full_conversation"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip(): return str(row[col])
    customer_cols = ["customer_text","user_text","client_text","khach_hang","customer"]
    agent_cols = ["agent_text","staff_text","employee_text","assistant_text","bot_text","nhan_vien","response"]
    cparts, aparts = [], []
    for col in customer_cols:
        if col in row and pd.notna(row[col]) and str(row[col]).strip(): cparts.append(str(row[col]))
    for col in agent_cols:
        if col in row and pd.notna(row[col]) and str(row[col]).strip(): aparts.append(str(row[col]))
    if cparts or aparts:
        text = ""
        if cparts: text += "Khách hàng: " + " ".join(cparts) + " "
        if aparts: text += "Nhân viên: " + " ".join(aparts)
        return text.strip()
    for col in TEXT_COL_CANDIDATES:
        if col in row and pd.notna(row[col]) and str(row[col]).strip(): return str(row[col])
    return ""

In [4]:
def create_toxicity_annotation_template():
    out_dir = Path("data/processed/toxicity"); out_dir.mkdir(parents=True, exist_ok=True)
    template_path = out_dir / "toxicity_annotation_template.csv"
    if not template_path.exists():
        cols = ["conversation_id","conversation_text","customer_text","agent_text","toxicity_label","toxicity_score","annotator","notes"]
        pd.DataFrame(columns=cols).to_csv(template_path, index=False, encoding="utf-8-sig")
    guide_path = out_dir / "TOXICITY_LABELING_GUIDE.md"
    guide_path.write_text("""# Hướng dẫn gán nhãn Binary Toxicity\n\nclean: không có ngôn ngữ độc hại/công kích/xúc phạm.\ntoxic: có ngôn ngữ tiêu cực, xúc phạm, công kích, chửi bới, đe dọa hoặc thù ghét.\n\n0 = clean\n1 = toxic\n""", encoding="utf-8")
    print("Template:", template_path)
    print("Guide:", guide_path)
create_toxicity_annotation_template()

Template: data\processed\toxicity\toxicity_annotation_template.csv
Guide: data\processed\toxicity\TOXICITY_LABELING_GUIDE.md


In [5]:
def load_toxicity_data_from_dirs(search_dirs, max_rows_per_file=None):
    rows = []
    for data_dir in search_dirs:
        if not data_dir.exists():
            print("Không tìm thấy folder:", data_dir); continue
        csv_files = list(data_dir.rglob("*.csv"))
        print(f"\nTìm thấy {len(csv_files)} file CSV trong {data_dir}")
        for csv_path in csv_files:
            name_lower = csv_path.name.lower()
            if any(x in name_lower for x in ["prediction","summary","dataset_info","annotation_template"]): continue
            try: df_temp = read_csv_robust(csv_path)
            except Exception as e:
                print(f"Không đọc được: {csv_path} | lỗi: {e}"); continue
            if max_rows_per_file is not None and len(df_temp) > max_rows_per_file:
                df_temp = df_temp.sample(max_rows_per_file, random_state=RANDOM_SEED)
            text_col = find_column(df_temp.columns, TEXT_COL_CANDIDATES)
            label_col = find_column(df_temp.columns, LABEL_COL_CANDIDATES)
            print("\n" + "="*100)
            print("FILE:", csv_path)
            print("Shape:", df_temp.shape)
            print("Columns:", list(df_temp.columns))
            print("Detected text_col :", text_col)
            print("Detected label_col:", label_col)
            if text_col is None:
                has_conv = any(c in df_temp.columns for c in ["conversation_text","customer_text","agent_text","user_text","response"])
                if not has_conv:
                    print("BỎ QUA: Không có cột text phù hợp."); continue
            if label_col is None:
                print("BỎ QUA: Không có cột label phù hợp."); continue
            print("Unique label raw values:")
            print(df_temp[label_col].value_counts(dropna=False).head(20))
            used = skipped_label = skipped_text = 0
            for _, row in df_temp.iterrows():
                text = build_toxicity_input(row)
                label = map_toxicity_label(row[label_col])
                if label is None:
                    skipped_label += 1; continue
                text = normalize_text(text)
                if len(text) < 2:
                    skipped_text += 1; continue
                rows.append({"text": text, "label": label, "source_file": str(csv_path)})
                used += 1
            print("Đã load:", used)
            print("Skip label:", skipped_label)
            print("Skip text:", skipped_text)
    data = pd.DataFrame(rows)
    if not data.empty: data = data.drop_duplicates(subset=["text","label"]).reset_index(drop=True)
    return data

def build_synthetic_vietnamese_toxicity_data():
    samples = [
        ("Khách hàng: Tôi muốn kiểm tra đơn hàng. Nhân viên: Dạ anh/chị cho em xin mã đơn hàng để em hỗ trợ kiểm tra ạ.", 0),
        ("Khách hàng: Tôi muốn đổi sản phẩm. Nhân viên: Dạ anh/chị gửi em hình ảnh sản phẩm và mã đơn hàng để em kiểm tra ạ.", 0),
        ("Khách hàng: Tôi chưa nhận được hàng. Nhân viên: Dạ em sẽ kiểm tra tình trạng vận chuyển giúp mình ạ.", 0),
        ("Khách hàng: Tôi cần hỗ trợ bảo hành. Nhân viên: Dạ anh/chị cho em xin thông tin sản phẩm để em hỗ trợ ạ.", 0),
        ("Khách hàng: Tôi muốn hỏi chính sách đổi trả. Nhân viên: Dạ em xin gửi anh/chị chính sách đổi trả hiện tại ạ.", 0),
        ("Khách hàng: Tôi chờ đơn hàng quá lâu rồi. Nhân viên: Không biết thì tự kiểm tra app đi, hỏi gì hỏi hoài vậy.", 1),
        ("Khách hàng: Tôi muốn đổi sản phẩm vì bị giao sai. Nhân viên: Bạn đặt ngu thì chịu, đừng có đổ lỗi cho bên tôi.", 1),
        ("Khách hàng: Tôi muốn hoàn tiền. Nhân viên: Phiền phức quá, gửi thông tin đi.", 1),
        ("Khách hàng: Tôi muốn gặp quản lý. Nhân viên: Im đi, đừng làm phiền nữa.", 1),
        ("Khách hàng: Tôi cần hỗ trợ đơn hàng. Nhân viên: Đồ ngu, tự xử lý đi.", 1),
    ]
    return pd.DataFrame([{"text": normalize_text(t), "label": y, "source_file": "synthetic_vietnamese_cskh"} for t, y in samples])

In [6]:
df_public = load_toxicity_data_from_dirs([TOXICITY_DATA_DIR], max_rows_per_file=MAX_ROWS_PER_FILE)
df_human = load_toxicity_data_from_dirs(LABELED_DATA_DIRS, max_rows_per_file=MAX_ROWS_PER_FILE)
df_synthetic = build_synthetic_vietnamese_toxicity_data() if USE_SYNTHETIC_CSKH else pd.DataFrame(columns=["text","label","source_file"])
print("\nDữ liệu public toxicity:", len(df_public))
print("Dữ liệu human-labeled:", len(df_human))
print("Dữ liệu synthetic:", len(df_synthetic))
df_list = []
if not df_public.empty: df_list.append(df_public)
if not df_human.empty: df_list.append(df_human)
if not df_synthetic.empty: df_list.append(df_synthetic)
if not df_list: raise ValueError("Không tìm thấy dữ liệu toxicity để train. Hãy kiểm tra folder data/raw/main_train/04_negative_attack_toxicity.")
df = pd.concat(df_list, ignore_index=True).drop_duplicates(subset=["text","label"]).reset_index(drop=True)
print("\nTổng số dòng dùng train:", len(df))
print("\nPhân bố nhãn:")
print(df["label"].map(id2label).value_counts())
print("\nPhân bố source_file:")
print(df["source_file"].value_counts().head(20))
display(df.head(10))


Tìm thấy 3 file CSV trong data\raw\main_train\04_negative_attack_toxicity

FILE: data\raw\main_train\04_negative_attack_toxicity\victsd\test.csv
Shape: (1000, 6)
Columns: ['Unnamed: 0', 'Comment', 'Constructiveness', 'Toxicity', 'Title', 'Topic']
Detected text_col : Comment
Detected label_col: Toxicity
Unique label raw values:
Toxicity
0    890
1    110
Name: count, dtype: int64
Đã load: 1000
Skip label: 0
Skip text: 0

FILE: data\raw\main_train\04_negative_attack_toxicity\victsd\train.csv
Shape: (7000, 6)
Columns: ['Unnamed: 0', 'Comment', 'Constructiveness', 'Toxicity', 'Title', 'Topic']
Detected text_col : Comment
Detected label_col: Toxicity
Unique label raw values:
Toxicity
0    6241
1     759
Name: count, dtype: int64
Đã load: 7000
Skip label: 0
Skip text: 0

FILE: data\raw\main_train\04_negative_attack_toxicity\victsd\validation.csv
Shape: (2000, 6)
Columns: ['Unnamed: 0', 'Comment', 'Constructiveness', 'Toxicity', 'Title', 'Topic']
Detected text_col : Comment
Detected label_co

,text,label,source_file
0,người ăn không hết kẻ lần chẳng ra,1,data\raw\main_train\04_negative_attack_toxicit...
1,nhiều người cứ nghĩ đạp xe là văn minh. haizzzz,1,data\raw\main_train\04_negative_attack_toxicit...
2,rất văn hoá,0,data\raw\main_train\04_negative_attack_toxicit...
3,đời ta ba mươi đời nó. mua chiếc xe cũng chỉ p...,0,data\raw\main_train\04_negative_attack_toxicit...
4,"tước bằng lái vĩnh viễn đi. chạy lếu láo thật,...",1,data\raw\main_train\04_negative_attack_toxicit...
5,cảm ơn các y bác sĩ,0,data\raw\main_train\04_negative_attack_toxicit...
6,thật tuyệt vời!,0,data\raw\main_train\04_negative_attack_toxicit...
7,quỷ dữ chứ không phải con người. thật xót xa c...,1,data\raw\main_train\04_negative_attack_toxicit...
8,rất nguy hiểm nếu không quản lý tôt nguồn gốc ...,0,data\raw\main_train\04_negative_attack_toxicit...
9,"nghe cũng nghẹn lòng thật, tương lai còn dài, ...",0,data\raw\main_train\04_negative_attack_toxicit...


In [7]:
label_counts = df["label"].value_counts().sort_index()
print("Label counts:")
for label_id, count in label_counts.items(): print(label_id, id2label[label_id], count)
missing = [i for i in range(2) if i not in label_counts.index]
if missing: print("\nCẢNH BÁO: Thiếu nhãn:", [(i, id2label[i]) for i in missing])
for label_id, label_name in id2label.items():
    print("\n" + "="*80)
    print("LABEL:", label_id, label_name)
    subset = df[df["label"] == label_id]
    display(subset.sample(min(5, len(subset)), random_state=RANDOM_SEED))

Label counts:
0 clean 8834
1 toxic 1106

LABEL: 0 clean


,text,label,source_file
4545,"ấn độ, trung quốc là những nước có nhiều người...",0,data\raw\main_train\04_negative_attack_toxicit...
6589,"làm ơn nâng tầm nhìn xa thiệt xa một chút dùm,...",0,data\raw\main_train\04_negative_attack_toxicit...
2716,phát triển là điều ai cũng muồn. nhưng cần có ...,0,data\raw\main_train\04_negative_attack_toxicit...
601,có kẻ thứ 3 cơ hội trong vách đá ở giữa.,0,data\raw\main_train\04_negative_attack_toxicit...
8866,thanh đồ long đao mà dương quá bỏ lại đây ư,0,data\raw\main_train\04_negative_attack_toxicit...



LABEL: 1 toxic


,text,label,source_file
2894,cải cách cái kiểu gì.mẫu giáo thì không cho vi...,1,data\raw\main_train\04_negative_attack_toxicit...
6820,"giới trẻ ngày nay ly hôn nhiều thật, đúng là c...",1,data\raw\main_train\04_negative_attack_toxicit...
565,"cái này là truy sát đến cùng rồi, lại không hề...",1,data\raw\main_train\04_negative_attack_toxicit...
6439,nói thật là nhìn thế nào thì cũng không thấy t...,1,data\raw\main_train\04_negative_attack_toxicit...
4417,đan trường ngoài vẻ ngoài điển trai thì hát ch...,1,data\raw\main_train\04_negative_attack_toxicit...


In [8]:
def balance_dataset(df, max_per_class=None):
    counts = df["label"].value_counts()
    print("Before balance:"); print(counts)
    if max_per_class is None: max_per_class = counts.min()
    parts = []
    for label_id in sorted(df["label"].unique()):
        part = df[df["label"] == label_id]
        if len(part) > max_per_class: part = part.sample(max_per_class, random_state=RANDOM_SEED)
        parts.append(part)
    out = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    print("\nAfter balance:"); print(out["label"].value_counts().sort_index())
    return out

# Nếu model bị lệch clean quá nhiều, bỏ comment dòng dưới:
# df = balance_dataset(df, max_per_class=None)

In [9]:
def split_data(df):
    if len(df) < 30: print("CẢNH BÁO: Dữ liệu rất ít. Nên thêm dữ liệu trước khi train thật.")
    counts = df["label"].value_counts()
    can_stratify = counts.min() >= 2 and len(counts) >= 2
    stratify_col = df["label"] if can_stratify else None
    train_df, temp_df = train_test_split(df, test_size=TEST_SIZE+VALID_SIZE, random_state=RANDOM_SEED, stratify=stratify_col)
    relative_test_size = TEST_SIZE/(TEST_SIZE+VALID_SIZE)
    if can_stratify and temp_df["label"].value_counts().min() >= 2: stratify_temp = temp_df["label"]
    else: stratify_temp = None
    valid_df, test_df = train_test_split(temp_df, test_size=relative_test_size, random_state=RANDOM_SEED, stratify=stratify_temp)
    print("Train:", len(train_df)); print("Valid:", len(valid_df)); print("Test :", len(test_df))
    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), test_df.reset_index(drop=True)
train_df, valid_df, test_df = split_data(df)
display(train_df.head())

Train: 6958
Valid: 1491
Test : 1491


,text,label,source_file
0,murray cố nhưng cũng k thể thi đấu đỉnh cao đc...,0,data\raw\main_train\04_negative_attack_toxicit...
1,sợ quá,0,data\raw\main_train\04_negative_attack_toxicit...
2,chắc các em đầu gấu này cũng muốn nghỉ học lâu...,1,data\raw\main_train\04_negative_attack_toxicit...
3,"tên đường kéo theo bao hệ luỵ. địa chỉ, sổ đất...",0,data\raw\main_train\04_negative_attack_toxicit...
4,phải trị thật nặng bọn người lưu manh cướp giậ...,1,data\raw\main_train\04_negative_attack_toxicit...


In [10]:
if MODEL_NAME == "vinai/phobert-base": tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
else: tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_dataset = Dataset.from_pandas(train_df[["text","label"]])
valid_dataset = Dataset.from_pandas(valid_df[["text","label"]])
test_dataset = Dataset.from_pandas(test_df[["text","label"]])
def tokenize_function(batch): return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)
train_dataset = train_dataset.remove_columns(["text"])
valid_dataset = valid_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format("torch"); valid_dataset.set_format("torch"); test_dataset.set_format("torch")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(train_dataset); print(valid_dataset); print(test_dataset)

Map: 100%|██████████| 1491/1491 [00:00<00:00, 6003.75 examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 6958
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 1491
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 1491
})


In [11]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id)
print(model.config)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 42135.54it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing f

RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "clean",
    "1": "toxic"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "clean": 0,
    "toxic": 1
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 258,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "tokenizer_class": "PhobertTokenizer",
  "transformers_version": "5.9.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 64001
}



In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision_weighted": precision, "recall_weighted": recall, "f1_weighted": f1}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
model.to(device)
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=valid_dataset, processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)
trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


Epoch,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
1,0.280864,0.279199,0.902750,0.887970,0.902750,0.877673
2,0.239623,0.308222,0.911469,0.899283,0.911469,0.898278
3,0.154621,0.351387,0.910127,0.901491,0.910127,0.904321


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.56it/s]


TrainOutput(global_step=2610, training_loss=0.2655290737919424, metrics={'train_runtime': 359.2984, 'train_samples_per_second': 58.097, 'train_steps_per_second': 7.264, 'total_flos': 1071507439622880.0, 'train_loss': 0.2655290737919424, 'epoch': 3.0})

In [13]:
print("Đánh giá trên tập test:")
test_result = trainer.evaluate(test_dataset)
print(test_result)
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids
print("\nCác nhãn thật có trong test:", sorted(set(labels.tolist())))
print("Các nhãn model dự đoán:", sorted(set(preds.tolist())))
print("\nClassification report:")
print(classification_report(labels, preds, labels=[0,1], target_names=[id2label[i] for i in range(2)], zero_division=0))
print("\nConfusion matrix:")
print(confusion_matrix(labels, preds, labels=[0,1]))
output_test_path = OUTPUT_DIR / "test_predictions.csv"
result_df = test_df.copy()
result_df["pred_label_id"] = preds
result_df["pred_label"] = [id2label[int(p)] for p in preds]
result_df["true_label"] = [id2label[int(y)] for y in labels]
result_df.to_csv(output_test_path, index=False, encoding="utf-8-sig")
final_dir = OUTPUT_DIR / "final_model"
final_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
with open(final_dir / "label_mapping.json", "w", encoding="utf-8") as f: json.dump({"id2label": id2label, "label2id": label2id}, f, ensure_ascii=False, indent=2)
print("Đã lưu prediction:", output_test_path)
print("Đã lưu model:", final_dir)
display(result_df.head(20))

Đánh giá trên tập test:


Training Loss,Validation Loss,Epoch,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
0.154621,0.368958,3,0.906103,0.898007,0.906103,0.901023


{'eval_loss': 0.3689582943916321, 'eval_accuracy': 0.9061032863849765, 'eval_precision_weighted': 0.8980065058487096, 'eval_recall_weighted': 0.9061032863849765, 'eval_f1_weighted': 0.901022591918526}



Các nhãn thật có trong test: [0, 1]
Các nhãn model dự đoán: [0, 1]

Classification report:
              precision    recall  f1-score   support

       clean       0.94      0.96      0.95      1325
       toxic       0.60      0.47      0.53       166

    accuracy                           0.91      1491
   macro avg       0.77      0.72      0.74      1491
weighted avg       0.90      0.91      0.90      1491


Confusion matrix:
[[1273   52]
 [  88   78]]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.59it/s]

Đã lưu prediction: models\toxicity_binary_phobert\test_predictions.csv
Đã lưu model: models\toxicity_binary_phobert\final_model


,text,label,source_file,pred_label_id,pred_label,true_label
0,đọc cảm nhận và hiểu thật sự cuốn hút hơn hình...,0,data\raw\main_train\04_negative_attack_toxicit...,0,clean,clean
1,pháp luật cũng như lương tâm thì chia đều cho ...,0,data\raw\main_train\04_negative_attack_toxicit...,0,clean,clean
2,"ủng hộ ý kiến này , họ hy sinh bản thân rất nh...",0,data\raw\main_train\04_negative_attack_toxicit...,0,clean,clean
3,"đề nghị pháp luật tước luôn quyền làm cha mẹ, ...",0,data\raw\main_train\04_negative_attack_toxicit...,0,clean,clean
4,em thấy các phụ huynh bảo phải đi học thêm ngo...,0,data\raw\main_train\04_negative_attack_toxicit...,0,clean,clean
5,xe chevrolet cruze lăn bánh 2016 mới đi 70k km...,0,data\raw\main_train\04_negative_attack_toxicit...,0,clean,clean
6,lại thêm một lời hứa không thể làm được hahaha,0,data\raw\main_train\04_negative_attack_toxicit...,0,clean,clean
7,"bài báo rất hay, mình cũng có một vài quan điể...",0,data\raw\main_train\04_negative_attack_toxicit...,0,clean,clean
8,"đài báo, tivi phải đồng lọat đưa tin, chính qu...",0,data\raw\main_train\04_negative_attack_toxicit...,0,clean,clean
9,một cái cmnd là chưa đủ ít nhất cũng phải có t...,0,data\raw\main_train\04_negative_attack_toxicit...,0,clean,clean


In [14]:
def predict_toxicity(text, model, tokenizer, id2label):
    model.eval()
    normalized = normalize_text(text)
    inputs = tokenizer(normalized, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        pred_id = int(torch.argmax(probs).item())
    pred_label = id2label[pred_id]
    score_map = {"clean": 0, "toxic": 3}
    return {"criterion": "negative_attack_toxicity", "label": pred_label, "score": score_map.get(pred_label), "confidence": float(probs[pred_id].item()), "probabilities": {id2label[i]: float(probs[i].item()) for i in range(len(id2label))}, "input": text}

test_texts = [
    "Khách hàng: Tôi muốn kiểm tra đơn hàng. Nhân viên: Dạ anh/chị cho em xin mã đơn hàng để em kiểm tra ạ.",
    "Khách hàng: Tôi chờ đơn hàng quá lâu rồi. Nhân viên: Không biết thì tự kiểm tra app đi, hỏi gì hỏi hoài vậy.",
    "Khách hàng: Tôi muốn đổi sản phẩm vì bị giao sai. Nhân viên: Bạn đặt ngu thì chịu, đừng có đổ lỗi cho bên tôi.",
    "Khách hàng: Tôi cần hỗ trợ đơn hàng. Nhân viên: Đồ ngu, tự xử lý đi.",
]
for text in test_texts:
    result = predict_toxicity(text, trainer.model, tokenizer, id2label)
    print("\n" + "="*80)
    print(text)
    print(json.dumps(result, ensure_ascii=False, indent=2))


Khách hàng: Tôi muốn kiểm tra đơn hàng. Nhân viên: Dạ anh/chị cho em xin mã đơn hàng để em kiểm tra ạ.
{
  "criterion": "negative_attack_toxicity",
  "label": "clean",
  "score": 0,
  "confidence": 0.9961091876029968,
  "probabilities": {
    "clean": 0.9961091876029968,
    "toxic": 0.003890871535986662
  },
  "input": "Khách hàng: Tôi muốn kiểm tra đơn hàng. Nhân viên: Dạ anh/chị cho em xin mã đơn hàng để em kiểm tra ạ."
}

Khách hàng: Tôi chờ đơn hàng quá lâu rồi. Nhân viên: Không biết thì tự kiểm tra app đi, hỏi gì hỏi hoài vậy.
{
  "criterion": "negative_attack_toxicity",
  "label": "clean",
  "score": 0,
  "confidence": 0.9930225014686584,
  "probabilities": {
    "clean": 0.9930225014686584,
    "toxic": 0.006977489683777094
  },
  "input": "Khách hàng: Tôi chờ đơn hàng quá lâu rồi. Nhân viên: Không biết thì tự kiểm tra app đi, hỏi gì hỏi hoài vậy."
}

Khách hàng: Tôi muốn đổi sản phẩm vì bị giao sai. Nhân viên: Bạn đặt ngu thì chịu, đừng có đổ lỗi cho bên tôi.
{
  "criterion":

In [15]:
while True:
    text = input("\nNhập hội thoại cần đánh giá toxic, hoặc nhập 'exit' để thoát:\n> ")
    if text.strip().lower() == "exit": break
    result = predict_toxicity(text, trainer.model, tokenizer, id2label)
    print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "criterion": "negative_attack_toxicity",
  "label": "toxic",
  "score": 3,
  "confidence": 0.9861193895339966,
  "probabilities": {
    "clean": 0.013880602084100246,
    "toxic": 0.9861193895339966
  },
  "input": "Im đi, loại khách như bạn thì biến khỏi đây, bên tôi không phục vụ loại người như bạn."
}
{
  "criterion": "negative_attack_toxicity",
  "label": "clean",
  "score": 0,
  "confidence": 0.9871721863746643,
  "probabilities": {
    "clean": 0.9871721863746643,
    "toxic": 0.012827830389142036
  },
  "input": "Bạn không biết tự kiểm tra trên app à? Hỏi gì hỏi hoài vậy, phiền phức quá"
}


## Ghi chú báo cáo

Đối với tiêu chí phát hiện ngôn ngữ tiêu cực hoặc công kích, dữ liệu ViCTSD được sử dụng cho bài toán phân loại nhị phân `clean/toxic`. Trường `Comment` được dùng làm văn bản đầu vào, còn trường `Toxicity` được dùng làm nhãn. Do ViCTSD không phân tách riêng mức `offensive` và `hate`, mô hình ở bước này chỉ đánh giá liệu nội dung có độc hại/công kích hay không. Nếu cần phân loại chi tiết `clean/offensive/hate`, cần bổ sung dataset có nhãn ba lớp như ViHSD.